In [0]:
# Databricks DQX Data Quality Job

# Install DQX on the cluster for this notebook run
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine
import yaml

# ---------------------------------------------------------
# Get environment parameter
# ---------------------------------------------------------

dbutils.widgets.text("catalog", "dqx_dev")

catalog = dbutils.widgets.get("catalog")

print(f"Running DQX against catalog: {catalog}")

# ---------------------------------------------------------
# Table names
# ---------------------------------------------------------

SOURCE_TABLE = f"{catalog}.customer.customer_raw"
SILVER_TABLE = f"{catalog}.customer.customer_silver"
QUARANTINE_TABLE = f"{catalog}.customer.customer_quarantine"

print(f"Source: {SOURCE_TABLE}")
print(f"Silver: {SILVER_TABLE}")
print(f"Quarantine: {QUARANTINE_TABLE}")

# ---------------------------------------------------------
# Read source table
# ---------------------------------------------------------

df = spark.table(SOURCE_TABLE)

print(f"Input record count: {df.count()}")

In [0]:
checks_custom = yaml.safe_load("""
- check:
    arguments:
      column: customer_name
      trim_strings: true
    function: is_not_null_and_not_empty
    criticality: error
    name: customer_name_not_null

- check:
    arguments:
      column: email
      trim_strings: true
    function: is_not_null_and_not_empty
    criticality: error
    name: email_not_null

- check:
    arguments:
      column: age
      max_limit: 100
      min_limit: 18
    function: is_in_range
    criticality: error
    name: age_range_check
""")

In [0]:
ws_client = WorkspaceClient()

dq_engine = DQEngine(ws_client)

silver_df, quarantine_df = (
    dq_engine.apply_checks_by_metadata_and_split(
        df,
        checks_custom
    )
)

In [0]:
print("Silver records:", silver_df.count())
print("Quarantine records:", quarantine_df.count())

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(QUARANTINE_TABLE)

print("DQX processing completed.")